# 06 Signals, thresholds and 7-1-7 — notebook

August in Santa Luz: the leptospirosis wave is fading, the standing water is not, and the *Aedes* mosquitoes are busy. Maya's question each Monday is simple: **is this week's dengue count unusual, or just the rainy season?** In this notebook you will:

1. read the weekly dengue series (`data/synthetic/ph/santa_luz_dengue_weekly.csv`, **synthetic**);
2. build an **epidemic threshold** — mean + 2 SD of the same weeks in ordinary years — and find the first week 2026 crosses it;
3. draw an **endemic channel**;
4. see why the latest weeks always look low (late reports);
5. measure **7-1-7 timeliness** on 40 past events (`signals_717.csv`) and read what slowed them down.

Every count here is invented; the methods are the real ones.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

## Step 1 — The weekly series

One row per **morbidity week**. This file uses ISO weeks (Monday to Sunday, labelled `2026-W32`); `iso_year` and `iso_week` are separate columns so we can line up "the same week" across years. Let's load it and look at 2026 so far.

In [ ]:
import statistics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

weekly = pd.read_csv("data/synthetic/ph/santa_luz_dengue_weekly.csv", parse_dates=["week_start"])
print(weekly.shape, "| years:", sorted(weekly["iso_year"].unique()))
print("Weeks per year:", weekly.groupby("iso_year").size().to_dict())

y2026 = weekly[weekly["iso_year"] == 2026].set_index("iso_week")
print("\n2026, weeks 28-39:")
print(y2026.loc[28:39, ["morbidity_week", "week_start", "cases"]].to_string())

## Step 2 — Build the threshold

The rule the CESU uses for this series (it is written in the data README):

* for ISO week *w*, take the counts of weeks *w*−1, *w* and *w*+1 in the **baseline years 2022–2025** — 4 years × 3 weeks = **12 numbers**;
* the threshold is their **mean + 2 standard deviations** (sample SD);
* 2019 is left out because it was an epidemic year, 2020–2021 because COVID-19 disrupted care-seeking and reporting. A baseline must describe *ordinary* years.

> **Ask Tala:** *I have a pandas DataFrame `weekly` with columns `iso_year`, `iso_week` (1–53) and `cases`. Made-up rows: `(2022, 31, 9)`, `(2022, 32, 12)`, `(2023, 32, 7)`. For every ISO week w from 1 to 53, compute an epidemic threshold = mean + 2 × sample SD of `cases` in weeks w−1, w, w+1 of the years 2022–2025 only (week 1's window wraps to week 52). Return a Series indexed by week. Then tell me the first 2026 week whose cases are above its threshold. Please keep it readable for a beginner.*

In [ ]:
BASELINE_YEARS = [2022, 2023, 2024, 2025]   # ordinary years only: not 2019, 2020, 2021, and never 2026


def window_weeks(week):
    # Weeks w-1, w, w+1; week 1 wraps to 52 and week 52 to 1 (week 53 uses 52, 53, 1)
    if week == 53:
        return {52, 53, 1}
    return {(week - 2) % 52 + 1, week, week % 52 + 1}


def epidemic_threshold(weekly, years):
    base = weekly[weekly["iso_year"].isin(years)]
    rows = {}
    for week in range(1, 54):
        values = base.loc[base["iso_week"].isin(window_weeks(week)), "cases"]
        rows[week] = {"n_values": len(values), "baseline_mean": values.mean(),
                      "threshold": values.mean() + 2 * values.std(ddof=1)}
    return pd.DataFrame(rows).T


thr = epidemic_threshold(weekly, BASELINE_YEARS)
season = y2026[["cases"]].join(thr)
season["above"] = season["cases"] > season["threshold"]
print(season.loc[28:39].round(1).to_string())

first_week = season.index[season["above"]].min()
print(f"\nFirst 2026 week above the threshold: W{first_week} "
      f"({y2026.loc[first_week, 'week_start']:%d %b}): {season.loc[first_week, 'cases']:.0f} cases "
      f"vs a threshold of {season.loc[first_week, 'threshold']:.1f}")

**Check it.** Three checks Doc Mon would do:

1. **By hand for one week.** List the 12 baseline values for W32 and compute mean + 2 SD with Python's `statistics` module — a second method, independent of pandas.
2. **The denominator of the window.** Every week must have exactly 12 values.
3. **The story.** No 2026 week before the first crossing is above the threshold, and every week after it is.

In [ ]:
base = weekly[weekly["iso_year"].isin(BASELINE_YEARS)]
w32_values = base.loc[base["iso_week"].isin([31, 32, 33]), "cases"].tolist()
print("The 12 baseline values for W32:", w32_values)
by_hand = statistics.mean(w32_values) + 2 * statistics.stdev(w32_values)
print(f"mean + 2 SD by hand: {by_hand:.2f}  | from the function: {thr.loc[32, 'threshold']:.2f}")

assert len(w32_values) == 12
assert abs(by_hand - thr.loc[32, "threshold"]) < 1e-9
assert (thr.loc[1:52, "n_values"] == 12).all()
assert not season.loc[:first_week - 1, "above"].any(), "no earlier week may be above"
assert season.loc[first_week:39, "above"].all()
print("All checks passed: the alert would have rung for W32, and stayed on through W39.")

## Step 3 — An endemic channel

An **endemic channel** draws what is "usual" for each week of the year so this year's curve can be read against it. We draw each baseline year as a thin grey line, the baseline mean (expected) and the threshold, then 2026 on top.

In [ ]:
weeks = np.arange(1, 53)
fig, ax = plt.subplots(figsize=(11, 4.2))
for year in BASELINE_YEARS:
    one = weekly[weekly["iso_year"] == year].set_index("iso_week")["cases"]
    ax.plot(one.index, one.values, color="#aeb7c0", linewidth=0.8)
ax.plot(weeks, thr.loc[weeks, "baseline_mean"], color="#2f7d32", linewidth=2, label="Baseline mean (2022-2025)")
ax.fill_between(weeks, 0, thr.loc[weeks, "threshold"], color="#2f7d32", alpha=0.10)
ax.plot(weeks, thr.loc[weeks, "threshold"], color="#b45309", linewidth=2, linestyle="--",
        label="Epidemic threshold (mean + 2 SD)")
bar_colours = np.where(season["above"], "#CE1126", "#0038A8")
ax.bar(season.index, season["cases"], color=bar_colours, alpha=0.85, label="2026 (red = above threshold)")
ax.annotate("Typhoon Amihan\n(W29)", xy=(29, 6), xytext=(18, 45), arrowprops={"arrowstyle": "->"})
ax.set_xlabel("ISO week")
ax.set_ylabel("Dengue cases per week")
ax.set_title("Santa Luz City (synthetic): 2026 dengue against its endemic channel")
ax.set_xlim(0.5, 52.5)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Tala got it wrong: the epidemic that raised its own bar

Maya's first prompt said "use all the years in the file". Tala did exactly that — and the threshold included **2026 itself** (and 2019, an epidemic year). An epidemic inside its own baseline pushes the mean and the SD up, so the bar rises with the curve. The cell below shows the effect, and the one-line `assert` that would have caught it.

In [ ]:
all_years = sorted(weekly["iso_year"].unique())          # Tala's version: 2019 ... 2026
thr_wrong = epidemic_threshold(weekly, all_years)
compare = y2026.loc[30:39, ["cases"]].copy()
compare["right_threshold"] = thr.loc[30:39, "threshold"]
compare["wrong_threshold"] = thr_wrong.loc[30:39, "threshold"]
compare["alarm_right"] = compare["cases"] > compare["right_threshold"]
compare["alarm_wrong"] = compare["cases"] > compare["wrong_threshold"]
print(compare.round(1).to_string())
print(f"\nWeeks W32-W39 flagged: right baseline {int(compare.loc[32:, 'alarm_right'].sum())} of 8, "
      f"Tala's baseline {int(compare.loc[32:, 'alarm_wrong'].sum())} of 8")
silenced = compare.index[compare["alarm_right"] & ~compare["alarm_wrong"]].tolist()
print(f"Bar for W32 rose from {thr.loc[32, 'threshold']:.1f} to {thr_wrong.loc[32, 'threshold']:.1f}; "
      f"alarm silent in weeks {silenced} while the epidemic was climbing")

# The guard that catches it: the season you are judging may never be in its own baseline
def check_baseline(years, season_year=2026, epidemic_years=(2019,)):
    assert season_year not in years, f"{season_year} is in its own baseline"
    assert not set(epidemic_years) & set(years), "an epidemic year is in the baseline"

check_baseline(BASELINE_YEARS)            # passes silently
try:
    check_baseline(all_years)
except AssertionError as err:
    print("Caught:", err)

## Step 4 — Why the latest weeks always look low

The weekly file counts cases by **week of onset**, but a case only reaches the CESU days later (consultation, then notification). So on any Wednesday, the week that just ended is not complete yet, and the week in progress looks almost empty. We can replay this with the line list from [chapter 04](../04_time_place_person.md): count dengue cases by onset week using only the rows **reported by** a given date.

In [ ]:
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv", parse_dates=["date_onset", "date_report"])
dengue = ll[ll["disease"] == "dengue"].copy()
dengue["onset_week"] = dengue["date_onset"].dt.isocalendar().week.astype(int)
delay_days = (dengue["date_report"] - dengue["date_onset"]).dt.days
print(f"Dengue onset -> report: median {delay_days.median():.0f} days, 90% within {delay_days.quantile(0.9):.0f} days")

snapshots = {}
for as_of in ["2026-09-02", "2026-09-16", "2026-09-30"]:
    known = dengue[dengue["date_report"] <= as_of]
    snapshots[f"known on {as_of[5:]}"] = known.groupby("onset_week").size()
snap = pd.DataFrame(snapshots).reindex(range(30, 40)).fillna(0).astype(int)
snap.index = [f"W{w}" for w in snap.index]
print(snap.T.to_string())

ax = snap.plot(kind="bar", rot=0, figsize=(10, 3.8), color=["#aeb7c0", "#FF9F1C", "#0038A8"], width=0.8)
ax.set_xlabel("Week of onset (2026)")
ax.set_ylabel("Dengue cases")
ax.set_title("The same weeks, seen on three different days")
plt.tight_layout()
plt.show()

## Step 5 — 7-1-7: how fast was the response?

**7-1-7** is a timeliness target for every public-health event:

* **7** — detect it within **7 days** of emergence (first case onset or event start);
* **1** — notify the public-health authority (here the PESU) within **1 day** of detection;
* **7** — complete the early response within **7 days** of notification.

`signals_717.csv` holds 40 synthetic events from January 2025 to August 2026, each with four dates and a `bottleneck` column that says what caused every missed target.

> **Ask Tala:** *I have a DataFrame `events` with date columns `date_emergence`, `date_detection`, `date_notification`, `date_response_complete` (made-up row: 2025-03-01, 2025-03-05, 2025-03-06, 2025-03-12). Add three columns with the delays in whole days (detection − emergence, notification − detection, response − notification), three True/False columns for the 7-1-7 targets (≤ 7, ≤ 1, ≤ 7) and one for "all three met". Then print how many events met each target, as counts and percentages.*

In [ ]:
events = pd.read_csv(
    "data/synthetic/ph/signals_717.csv",
    parse_dates=["date_emergence", "date_detection", "date_notification", "date_response_complete"],
)
events["days_to_detect"] = (events["date_detection"] - events["date_emergence"]).dt.days
events["days_to_notify"] = (events["date_notification"] - events["date_detection"]).dt.days
events["days_to_respond"] = (events["date_response_complete"] - events["date_notification"]).dt.days

events["detect_ok"] = events["days_to_detect"] <= 7
events["notify_ok"] = events["days_to_notify"] <= 1
events["respond_ok"] = events["days_to_respond"] <= 7
events["all_ok"] = events[["detect_ok", "notify_ok", "respond_ok"]].all(axis=1)

targets = ["detect_ok", "notify_ok", "respond_ok", "all_ok"]
summary = pd.DataFrame({
    "met": events[targets].sum(),
    "missed": (~events[targets]).sum(),
    "percent_met": (events[targets].mean() * 100).round(0),
})
print(summary.to_string())
print("\nMedian delays (days):", events[["days_to_detect", "days_to_notify", "days_to_respond"]].median().to_dict())

**Check it.** Count one event by hand. EV-040 is the story's dengue surge in Barangay Baybay: emergence on the Monday of the threshold week. Read its four dates, subtract them on paper, and compare with Tala's columns. Then check that "all three" can never be larger than any single target.

In [ ]:
ev = events.set_index("event_id").loc["EV-040"]
print(ev[["disease", "barangay", "date_emergence", "date_detection", "date_notification", "date_response_complete"]])
print("Delays:", ev["days_to_detect"], ev["days_to_notify"], ev["days_to_respond"])

assert ev["days_to_detect"] == (pd.Timestamp("2026-08-10") - pd.Timestamp("2026-08-03")).days == 7
assert ev["detect_ok"] and ev["notify_ok"] and not ev["respond_ok"]
assert summary.loc["all_ok", "met"] <= summary.loc[["detect_ok", "notify_ok", "respond_ok"], "met"].min()
assert (events[["days_to_detect", "days_to_notify", "days_to_respond"]] >= 0).all().all(), "a negative delay = a date typo"
print("Checks passed.")

## Step 6 — Read the bottleneck column

Percentages tell you **how often** a target was missed; the `bottleneck` column tells you **why**. One event can have several causes joined with `"; "`, so we split them, count them, and look at which target each missed event failed. This is the list Maya brings to the Local Health Board: problems that money, a back-up DSO or a memo can fix.

In [ ]:
missed = events[events["bottleneck"] != "none"]
causes = missed["bottleneck"].str.split("; ").explode().str.strip()
print(f"{len(missed)} events missed at least one target; {len(causes)} causes recorded.\n")
print(causes.value_counts().head(6).to_string())

print("\nMissed per target:", (~events[["detect_ok", "notify_ok", "respond_ok"]]).sum().to_dict())
print("\nThe two Santa Luz events of 2026:")
print(events.tail(2)[["event_id", "disease", "barangay", "days_to_detect", "days_to_notify",
                      "days_to_respond", "bottleneck"]].to_string(index=False))

## Your turn

1. **A stricter bar.** Recompute the threshold as mean + **1.5** SD (copy `epidemic_threshold` and change one number). Which 2026 week is first above it now? What would that cost in false alarms in 2022–2025? *(Hint: join each baseline year to `thr` the same way as `season` and count the weeks above.)*
2. **Same week only.** Use only week *w* (not *w*−1, *w*, *w*+1) of the baseline years. How many values per week is that, and does the first crossing move? *(Hint: `window_weeks` is the only thing to change.)*
3. **Which disease is slowest?** Group `events` by `disease` and compute the median `days_to_detect`. Which disease would you train BHWs to spot first?

In [ ]:
# 1. Stricter bar (starter)
def epidemic_threshold_k(weekly, years, k):
    base = weekly[weekly["iso_year"].isin(years)]
    out = {}
    for week in range(1, 54):
        values = base.loc[base["iso_week"].isin(window_weeks(week)), "cases"]
        out[week] = values.mean() + k * values.std(ddof=1)
    return pd.Series(out)

thr_15 = epidemic_threshold_k(weekly, BASELINE_YEARS, 1.5)
print("First 2026 week above mean + 1.5 SD:", (y2026["cases"] > thr_15.loc[y2026.index]).idxmax())

# 3. Slowest disease to detect (starter)
print(events.groupby("disease")["days_to_detect"].median().sort_values(ascending=False).head(3))

## What you did

* Built an epidemic threshold from **ordinary** baseline years only, checked it by hand, and found the first crossing (2026-W32).
* Drew an endemic channel and caught a baseline that contained the epidemic itself.
* Saw how late reports make the latest weeks look low — read the newest bars as "at least".
* Measured 7-1-7 and turned the `bottleneck` column into a to-do list.

Next: [chapter 07](../07_forecast_timesfm.md) asks what the next eight weeks may bring.